# Welcome to Graph Rewrite

**Graph Rewrite** is a Python library for performing graph transformations using a declarative approach. It provides a framework for transforming graphs by finding and rewriting subgraphs that match specific patterns. This approach combines structure-based matching with attribute filtering and advanced filtering options, making it suitable for rewriting subgraphs in ASTs, execution graphs, term graphs, and other graph types to improve efficiency.

The library includes extended features that go beyond standard graph rewriting, enabling the efficient identification of complex patterns within input graphs.

## Installation

```bash
#Install package given a conda environment with python>3.9
git clone https://github.com/DeanLight/graph_rewrite
cd graph_rewrite
pip install -e .

```

## Docker

```bash
cd graph_rewrite
# build container
docker-compose build

# spin up the container
docker-compose up

# get a bash terminal on a spun up container
docker-compose exec main bash

# spin up and get a bash terminal (closing it will close the container)
docker-compose run main bash

```

## Getting started

In [178]:
import networkx as nx
from graph_rewrite.transform import rewrite_iter
from graph_rewrite.transform import rewrite

### The Rewrite Function

The library interface is built around a single main function, `rewrite`. This function lets you define patterns, find matches, and transform subgraphs in an input graph according to specific rules you set.

To use `rewrite`, you provide three pattern strings: *LHS*, *P*, and *RHS*. These strings describe the patterns you want to match in the input graph and the transformations to apply. You can also add optional parameters to customize how matches are found and transformed.

This interface is flexible enough to support a wide range of graph transformation needs.

### Using LHS, P, and RHS Patterns in Graph Rewrite

The `rewrite` function in the Graph Rewrite library uses three key components to define and apply transformations to graphs: **LHS**, **P**, and **RHS** patterns. These patterns work together to match and rewrite subgraphs within the input graph.

- **LHS (Left-Hand Side)**: Defines the subgraph structure we’re searching for in the input graph. This represents the "before" state and specifies the exact pattern we’re looking to match prior to rewriting.
- **P (Preserve)**: Defines which parts of the matched subgraph should remain unchanged after the rewrite. This can include specific nodes, edges, and their attributes. If an empty P string is passed to the *rewrite* function, no objects from the matched subgraph will be removed.
- **RHS (Right-Hand Side)**: Defines the new structure to create based on the matched subgraph, allowing us to modify or add elements within the graph.

## Usecases 

Here are some simple use cases that demonstrate how LHS, P, and RHS work together to define different transformations:

#### 1. Basic Attribute Replacement

We start with an input directed graph (`input_graph`) containing a single node `'A'` with a label attribute set to `'Node_A'`.

Using the following strings:
* **LHS**: `lhs = 'x[label="Node_A"]'`, specifying a search for any node with a label attribute equal to `'Node_A'`. Here, `'x'` is a symbolic name for the matching node.
* **P**: `p = 'x[label]'`, indicating that we want to preserve the node `'x'` and retain its label attribute. This prevents the node from being removed during the rewrite but allows its label value to be updated.
* **RHS**: `rhs = 'x[label="Node_B"]'`, specifying that we want to change the label of the node `'x'` to `'Node_B'`.

We call `rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)` to apply the transformation. This finds the node `'A'` with `label="Node_A"`, preserves it, and updates the label to `"Node_B"`. 

After the rewrite, `input_graph.nodes(data=True)` should print `[('A', {'label': 'Node_B'})]`, confirming that the label of node `'A'` was successfully updated to `'Node_B'`.

In [179]:
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A', {'label': 'Node_A'})])
lhs = 'x[label="Node_A"]'
p = 'x[label]'
rhs = 'x[label="Node_B"]'
rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)
print(input_graph.nodes(data=True)) # Should print [('A', {'label': 'Node_B'})]

[('A', {'label': 'Node_B'})]


#### 2. Removing an Intermediate Node and Adding a Direct Edge 

In this test, we start with a directed graph (`input_graph`) that contains a path `A -> B -> C`. Using the LHS, P, and RHS strings, we’ll find this pattern, remove the intermediate node `B`, and add a direct edge from `A` to `C`.

We use the following strings:
* **LHS**: The string `lhs = 'x->y->z'` specifies that we’re looking for a path where node `x` connects to node `y`, which in turn connects to node `z`. Here, `x`, `y`, and `z` are symbolic names for the matching nodes in the input graph.
* **P**: The string `p = 'x,z'` specifies that we want to preserve the direct path from `x` to `z`, which will be created during the rewrite. This also implies that `y` is removed from the graph during the rewrite since it’s not part of `P`.
* **RHS**: The string `rhs = 'x->z'` specifies that we want to create a new edge directly from `x` to `z` after removing `y`.

By calling `rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)`, we apply the transformation. The function will find the path `A -> B -> C`, remove the intermediate node `B`, and create a direct edge from `A` to `C`.

After the rewrite, `input_graph.edges()` should print `[('A', 'C')]`, showing that the transformation successfully removed `B` and added the edge `A -> C`.

In [180]:
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A', {'label': 'Node_A'}), ('B', {'label': 'Node_B'}), ('C', {'label': 'Node_C'})])
input_graph.add_edges_from([('A', 'B'), ('B', 'C')])
lhs = 'x->y->z'
p = 'x,z'
rhs = 'x->z'
rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)
print(list(input_graph.edges()))  # Should print [('A', 'C')]

[('A', 'C')]


#### 3. Finding and Transforming a Parent-Child Pair with Attribute Modification

In this example, we start with a graph where each parent node connects to a child node. The goal is to find all such pairs where the parent node has an attribute `'val'`, increment this `'val'` by 1, and store the result in the connecting edge.

We use the following strings:
* **LHS**: The string `lhs = 'a[val]->b'` specifies that we are looking for an edge from node `'a'` to node `'b'` where `'a'` has an attribute `'val'`.
* **P**: The string `p = 'a->b'` specifies that we want to preserve both nodes `'a'` and `'b'` and the edge between them.
* **RHS**: The string `rhs = 'a-[val={{new_val}}]->b'` specifies that we want to add an edge attribute `'val'` to the edge from `'a'` to `'b'`, which is calculated by incrementing the existing `'val'` attribute of node `'a'` by 1.

We also provide a custom `render_rhs` function to calculate the new value for the edge. This function uses the value of `'val'` in `'a'`, increments it by 1, and stores it in the edge between `'a'` and `'b'`.

After applying `rewrite`, the transformed graph should have the edge from `'a'` to `'b'` updated with the new `'val'` attribute.

In [181]:
input_graph = nx.DiGraph()
input_graph.add_node('A', val=10)  # Example initial value
input_graph.add_node('B')
input_graph.add_edge('A', 'B')
lhs = 'a[val]->b'
p = 'a->b'
rhs = 'a-[val={{new_val}}]->b'
rewrite(
    input_graph=input_graph,
    lhs=lhs,
    p=p,
    rhs=rhs,
    render_rhs={'new_val': lambda match: match['a']['val'] + 1}
)
print(input_graph.edges(data=True))  # Should print [('A', 'B', {'val': 11})]

[('A', 'B', {'val': 11})]


### 4. Using a Collections Sub-Pattern To Create a List of All Grandchildren and Print Their Labels

This example demonstrates how to capture multiple input nodes that match a collection pattern node and apply an imperative side effect. We use a helper function to both gather the labels of each grandchild node and print the list of grandchildren as a side effect.

Starting with a graph containing nodes `'A'`, `'B'`, `'C'`, `'D'`, and `'E'`, connected as follows: `A -> B`, `A -> C`, `B -> D`, and `B -> E`.

We define the transformation using the following patterns:
* **LHS**: `lhs = 'x->y;y->z'` specifies a pattern where **x** connects to **y**, and **y** has at least one outgoing edge to **z**. Here, **x** and **y** represent unique nodes in each match, while **z** represents a collection of nodes connected to **y**.
* **P**: `p = 'x->y,y->z'` specifies that we want to preserve the edges from **x** to **y** and from **y** to each **z** node in the collection.
* **RHS**: `rhs = 'x[grandchildren={{new_val}}]->y,y->z'` specifies that we want to add an attribute `grandchildren` to **x**. This attribute will be a list containing the labels of all nodes in the **z** collection, captured by `{{new_val}}`.

We use a helper function, `_calculate_and_print_grandchildren`, in `render_rhs` to gather the labels of each **z** node in the collection. This function has an imperative side effect: it prints the list of grandchildren labels each time a match is processed. The function then returns the list of labels to be stored in the `grandchildren` attribute of **x**.

In [182]:
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A', {'label': 'Node_A'}), ('B', {'label': 'Node_B'}), ('C', {'label': 'Node_C'}), ('D', {'label': 'Node_D'}), ('E', {'label': 'Node_E'})])
input_graph.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'D'), ('B', 'E')])

lhs = 'x->y;y->z'
p = 'x->y,y->z'
rhs = 'x[grandchildren={{new_val}}]->y,y->z'
render_rhs={'new_val': lambda match: _calculate_and_print_grandchildren(match)}

# Helper function that helps us both calculate the new value, and have an imperative side effect of printing the grandchildren.
def _calculate_and_print_grandchildren(match):
    grandchildren = match['z']['label']
    print(f"the grandchildren of {match['x']['label']} are: {grandchildren}")
    return grandchildren

# Should print: the grandchildren of Node_A are: Node_D, Node_E
rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs, render_rhs={'new_val': lambda match: _calculate_and_print_grandchildren(match)})

the grandchildren of Node_A are: ['Node_E', 'Node_D']


### 5. Counting Matches

In this example, we use `rewrite_iter` to count the number of matches in an input graph and store this count in an external counter object. This approach showcases how `rewrite_iter` can perform side effects outside the graph itself.

We start with a simple directed graph (input_graph) containing nodes `'A'`, `'B'` and `'C'` connected as follows: `A -> B -> C`.

We define the transformation with the following patterns:
* **LHS**: `lhs = 'x->y'` specifies a pattern where node **x** connects to node **y**. Here, **x** and **y** represent unique nodes in each match
* **P**: Not used, specifies that we want to preserve both nodes **x** and **y** and the edge between them, preventing these nodes and edge from being removed.
* **RHS**: Not used, simply specifies that we want to retain the same structure without making any changes to the graph.

As `rewrite_iter` processes each match, we increment a counter in an external `match_counter` object to track how many matches were found.

In [183]:
input_graph = nx.DiGraph()
input_graph.add_edges_from([('A', 'B'), ('B', 'C')])

# External counter to track the number of matches
match_counter = {'count': 0}

# Define patterns
lhs = 'x->y'

# Use rewrite_iter and update the counter
for match in rewrite_iter(input_graph=input_graph, lhs=lhs):
    match_counter['count'] += 1

print(match_counter['count'])  # Should print 2, as there are two matches: A -> B and B -> C
# Expected output: 2

2


## Example 6: Collecting Labels of Matched Nodes

In this example, we use `rewrite_iter` to gather the labels of all matched nodes into an external list. We start with a graph that contains three nodes:`'A'`, `'B'` and `'C'`, each with a label attribute. The goal is to find all nodes connected by a direct edge, and add their labels to an external list.

We define the transformation using the following patterns:
* **LHS**: `lhs = 'x[label]->y'` specifies that we are looking for an edge where the starting node **x** has a `label` attribute and connects to node **y**.
* **P**: Not used, specifies that we want to preserve both nodes **x** and **y** and the edge between them, as well as the `label` attribute of node **x**, preventing these from being removed.
* **RHS**: Not used, simply specifies that we want to retain the same structure without making any changes to the graph.

As `rewrite_iter` finds matches, it appends the labels of the matching node x to an external list `matched_labels`.

In [184]:
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A', {'label': 'Label_A'}), ('B', {'label': 'Label_B'}), ('C', {'label': 'Label_C'})])
input_graph.add_edges_from([('A', 'B'), ('B', 'C')])

# External list to collect labels of matched nodes
matched_labels = []

# Define patterns
lhs = 'x[label]->y'

# Use rewrite_iter and update the external list
for match in rewrite_iter(input_graph=input_graph, lhs=lhs):
    matched_labels.append(match['x']['label'])

print(matched_labels)  # Should print ['Label_A', 'Label_B'], as nodes A and B are the matched nodes with labels
# Expected output: ['Label_A', 'Label_B']

['Label_B', 'Label_A']


### 7. Simulating Leaf Node Removal Effect on an Out-Degree Histogram

In this example, we use `rewrite_iter` to analyze the impact of removing "leaf" nodes (nodes with an out-degree of 0) without actually deleting them from the graph. Instead, we build an out-degree histogram and simulate what the histogram would look like if all leaf nodes were removed, and their parent nodes' out-degrees were adjusted accordingly.

We start with a directed graph (`input_graph`) containing nodes `A`, `B`, `C`, `D`, and `E`, connected as follows: `A -> B -> C`, `A -> D`, and `B -> E`. Our goal is to:
1. Build an initial histogram of node out-degrees.
2. Simulate the effect of removing the leaf nodes `C`, `D`, and `E` by adjusting the out-degrees in the histogram, without modifying the graph structure.
3. Display both the initial histogram and the simulated histogram to show how the out-degree distribution would change if these nodes were actually removed.

The steps are as follows:

1. **Build Initial Out-Degree Histogram**: We first initialize each node with its actual out-degree in the graph and build a histogram representing the current out-degree distribution.
  
2. **Simulate Leaf Node Removal**:
    - For each parent node connected to a leaf node (identified using `rewrite_iter` with `lhs='x; x->y[out_degree=0]'`), calculate how the out-degree of the parent would change if its leaf nodes were removed.
    - Update the histogram as if the leaf nodes were removed, adjusting the out-degree of each parent node accordingly.
    
This simulation provides a "what-if" analysis of the graph's structure, allowing us to see how the out-degree distribution would change without modifying the graph itself.

In [185]:
from collections import defaultdict

# Create the input graph
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A'), ('B'), ('C'), ('D'), ('E')])
input_graph.add_edges_from([('A', 'B'), ('B', 'C'), ('A', 'D'), ('B', 'E')])

# Initialize the out-degree attribute for each node: A and B have 2 out-degree, C, D and E have 0 out-degree
for node in input_graph.nodes(data=True):
    node[1]['out_degree'] = input_graph.out_degree(node[0])

# Build the initial histogram of out-degrees
out_degree_histogram = defaultdict(int)
for _, data in input_graph.nodes(data=True):
    out_degree_histogram[data['out_degree']] += 1

# Copy the initial histogram to simulate the removal of leaf nodes, and remove the 0 out-degree nodes (leaves)
simulated_histogram = out_degree_histogram.copy()
del simulated_histogram[0]

# Apply rewrite to find all parents of any leaf node, and calculate the simulated out-degree histogram
for match in rewrite_iter(input_graph=input_graph, lhs='x;x->y[out_degree=0]'):
    number_of_leaf_children = len(match['y']._get_nodes())

    # Update the simulated histogram value for the out-degree of the parent node
    simulated_histogram[match['x']['out_degree']] -= 1
    if simulated_histogram[match['x']['out_degree']] == 0:
        del simulated_histogram[match['x']['out_degree']]

    # Update the simulated histogram value for the out-degree of the parent node after the deletion of the leaf nodes
    simulated_histogram[match['x']['out_degree'] - number_of_leaf_children] += 1

# Print the initial and simulated histograms 
print("Initial Out-Degree Histogram:")
print(dict(out_degree_histogram)) # Should print {2: 2, 0: 3}, as nodes A and B have 2 out-degree and nodes C, D and E have 0 out-degree

print("\nSimulated Out-Degree Histogram (if leaf nodes were removed):")
print(dict(simulated_histogram)) # Should print {1: 1, 0: 1}, as after the deletion of C, D and E, node A would have one child and node B would have no children

Initial Out-Degree Histogram:
{2: 2, 0: 3}

Simulated Out-Degree Histogram (if leaf nodes were removed):
{0: 1, 1: 1}
